In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Watches_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Software_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_PC_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Music_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Video_DVD_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Beauty_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Mobile_Electronics_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Shoes_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Apparel_v1_00.tsv
/kaggle/input/datasets/c

In [2]:
!pip install pyspark -q

In [3]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:10]:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Watches_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Software_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_PC_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Music_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Video_DVD_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Beauty_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Mobile_Electronics_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Shoes_v1_00.tsv
/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/amazon_reviews_us_Apparel_v1_00.tsv
/kaggle/input/datasets/c

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [5]:
spark = SparkSession.builder \
    .appName("AmazonReviews") \
    .config("spark.driver.memory", "24g") \
    .config("spark.executor.memory", "24g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 11:02:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Đọc toàn bộ dữ liệu từ 37 file csv của dataset:, 37 file csv chứa review chia theo từng category ung voi ten file; cau truc chung cua cac file:

| Cột | Tên tiếng Việt | Ý nghĩa |
|----|----|----|
| marketplace | Thị trường | Nơi đánh giá được đăng |
| customer_id | Mã khách hàng | ID của khách hàng viết đánh giá |
| review_id | Mã đánh giá | ID duy nhất cho mỗi bài đánh giá |
| product_id | Mã sản phẩm | ID của sản phẩm được đánh giá |
| product_parent | Mã nhóm sản phẩm | ID của nhóm sản phẩm cha (các biến thể như màu, kích thước) |
| product_title | Tên sản phẩm | Tên đầy đủ của sản phẩm |
| product_category | Danh mục sản phẩm | Loại sản phẩm (Books, Electronics, Watches...) |
| star_rating | Số sao đánh giá | Điểm đánh giá từ **1 đến 5 sao** |
| helpful_votes | Số lượt đánh giá hữu ích | Số người thấy review này hữu ích |
| total_votes | Tổng số lượt bình chọn | Tổng số người đã vote (bao gồm hữu ích và không hữu ích) |
| vine | Chương trình Vine | Review có thuộc chương trình **Amazon Vine** hay không (Y/N) |
| verified_purchase | Mua hàng xác thực | Người review có thực sự mua sản phẩm trên Amazon hay không (Y/N) |
| review_headline | Tiêu đề đánh giá | Tiêu đề ngắn của bài review |
| review_body | Nội dung đánh giá | Nội dung chi tiết của review |
| review_date | Ngày đánh giá | Ngày người dùng đăng review |

In [6]:
data_path = "/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/"

df_raw = spark.read \
    .option("sep", "\t") \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv(data_path)

Schema chung của dataset, bao gồm các cột + kiểu dữ liệu

In [7]:
df_raw.printSchema()

root
 |-- marketplace: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: string (nullable = true)
 |-- helpful_votes: string (nullable = true)
 |-- total_votes: string (nullable = true)
 |-- vine: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: string (nullable = true)



In [8]:
total_rows = df_raw.count()  
print(f"Total rows: {total_rows:,}")




Total rows: 109,532,002


Ép kiểu và loại bỏ date không hợp lệ

In [9]:
from pyspark.sql.functions import col
df_clean = df_raw.filter(
    col("review_date").rlike(r"^\d{4}-\d{2}-\d{2}$")
).withColumn(
    "star_rating",
    col("star_rating").cast("int")
).withColumn(
    "helpful_votes",
    col("helpful_votes").cast("int")
).withColumn(
    "total_votes",
    col("total_votes").cast("int")
).withColumn(
    "review_date",
    to_date("review_date", "yyyy-MM-dd")
)


In [10]:
df_clean.printSchema()

root
 |-- marketplace: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- vine: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: date (nullable = true)



In [11]:
from pyspark.sql.functions import *

validation = df_clean.select(
    count("*").alias("total_rows"),
    count(when(col("review_id").isNull(), True)).alias("null_review_id"),
    count(when(col("product_id").isNull(), True)).alias("null_product_id"),
    count(when(col("star_rating").isNull(), True)).alias("null_star_rating"),
    count(when(col("review_body").isNull(), True)).alias("null_review_body"),
    count(when((col("star_rating") < 1) | (col("star_rating") > 5), True)).alias("invalid_rating"),
    count(when(col("helpful_votes") > col("total_votes"), True)).alias("vote_inconsistency"),
    count(when((col("review_body") == "") | col("review_body").isNull(), True)).alias("empty_reviews"),
    count(when(length("review_body") < 10, True)).alias("short_reviews"),
    min("review_date").alias("min_date"),
    max("review_date").alias("max_date")
)

validation.show(vertical=True)
df_clean.groupBy("product_category") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(100, False)

-RECORD 0------------------------
 total_rows         | 109526510  
 null_review_id     | 0          
 null_product_id    | 0          
 null_star_rating   | 0          
 null_review_body   | 5167       
 invalid_rating     | 0          
 vote_inconsistency | 0          
 empty_reviews      | 5167       
 short_reviews      | 3459626    
 min_date           | 1995-06-24 
 max_date           | 2015-08-31 



+------------------------+-------+
|product_category        |count  |
+------------------------+-------+
|Wireless                |9014075|
|PC                      |6963652|
|Mobile_Apps             |6475147|
|Digital_Ebook_Purchase  |6342335|
|Video DVD               |6142393|
|Apparel                 |5881886|
|Music                   |5517180|
|Health & Personal Care  |5313710|
|Beauty                  |5094281|
|Digital_Video_Download  |5049782|
|Toys                    |4917040|
|Sports                  |4837038|
|Shoes                   |4366141|
|Books                   |3941313|
|Automotive              |3511080|
|Electronics             |3102421|
|Office Products         |2642527|
|Pet Products            |2639848|
|Grocery                 |2393332|
|Outdoors                |2302982|
|Camera                  |1817747|
|Video Games             |1795638|
|Digital_Music_Purchase  |1788865|
|Baby                    |1754995|
|Tools                   |1747500|
|Watches            

loại duplicate

In [12]:
df_clean = df_clean.dropDuplicates(["review_id"])

Doi verified_purchase, vine ve kieu du lieu so

In [13]:
df_clean = df_clean.withColumn("vine", (col("vine") == "Y").cast("int")) \
       .withColumn("verified_purchase", (col("verified_purchase") == "Y").cast("int"))

In [14]:
keep_categories = [
    "Wireless",
    "PC",
    "Video DVD",
    "Apparel",
    "Music",
    "Health & Personal Care",
    "Beauty",
    "Toys",
    "Sports",
    "Shoes",
    "Books",
    "Automotive",
    "Electronics",
    "Office Products",
    "Pet Products",
    "Grocery",
    "Outdoors",
    "Camera",
    "Video Games",
    "Baby",
    "Tools"
]


df = df_clean.select(
    "review_id",
    "customer_id",
    "product_id",
    "product_parent",
    "product_title",
    "product_category",
    "star_rating",
    "helpful_votes",
    "total_votes",
    "vine",
    "verified_purchase",
    "review_headline",
    "review_body",
    "review_date"
)

df = df.filter(col("product_category").isin(keep_categories))

df = df.filter(col("review_body").isNotNull())

df = df.withColumn("helpful_votes", coalesce(col("helpful_votes"), lit(0))) \
       .withColumn("total_votes", coalesce(col("total_votes"), lit(0)))

df = df.withColumn(
    "helpful_ratio",
    when(col("total_votes") > 0,
         col("helpful_votes") / col("total_votes"))
    .otherwise(0)
).withColumn("review_length", length("review_body"))

In [15]:
df.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- helpful_votes: integer (nullable = false)
 |-- total_votes: integer (nullable = false)
 |-- vine: integer (nullable = true)
 |-- verified_purchase: integer (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- helpful_ratio: double (nullable = true)
 |-- review_length: integer (nullable = true)



In [16]:
cap = 3000000
counts = df.groupBy("product_category").count().collect()

counts_dict = {row["product_category"]: row["count"] for row in counts}

fractions = {
    k: 1.0 if v <= cap else cap / v
    for k, v in counts_dict.items()
}

df_balanced = df.sampleBy(
    "product_category",
    fractions=fractions,
    seed=42
)

In [17]:
df_balanced = df.sampleBy(
    "product_category",
    fractions=fractions,
    seed=42
)

In [18]:
df_balanced.write \
    .mode("overwrite") \
    .partitionBy("product_category") \
    .parquet("/kaggle/working/amazon_reviews_balanced/")

print("Saved to Parquet successfully")


26/03/16 11:33:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Saved to Parquet successfully
